# The Duwamish Computer — inside the machine

This notebook opens up the Duwamish, the balanced-ternary computer in this
repository, and shows it working, cycle by cycle.

Every animation here is **exact**. The notebook runs the real Model 30
micro-engine one micro-cycle at a time and records everything the console
would have shown: every register, the condition trit, the mode, the
micro-program counter, the control lines asserted, every read and write of
core, traps and the printer. The front panel then replays that recording,
so the lamps light in exactly the order the machine lit them.

**How to use the front panels.** Run each cell (or *Run All*). In a panel:

| control | does |
|---|---|
| **▶ run** / **❚❚ pause** | play the recording at the chosen speed |
| **step µ** | one micro-cycle |
| **step instr** | on to the start of the next instruction |
| **◀◀** / **⏮** | back one micro-cycle / back to the start |
| slider | scrub anywhere in the recording |
| speed | from 1 to 10,000 micro-cycles a second |

Lamps: **amber is +1, blue is −1, dark is 0**, which are the three values of a
trit. A register's row label turns amber when its value changed in the
micro-cycle just shown.

Needs only Python 3 and Jupyter (Notebook, Lab or VS Code); no extra packages.

In [1]:
import os, sys
# make the duwamish package importable from the repository
here = os.path.abspath(os.getcwd())
while here != os.path.dirname(here) and not os.path.isdir(os.path.join(here, "duwamish")):
    here = os.path.dirname(here)
sys.path.insert(0, here)
os.chdir(here)

from duwamish import panel, ternary, isa, microasm, machine, satellite, salish, triad, fpu
print("Duwamish loaded from", here)

Duwamish loaded from /home/user/Duwamish


## 1. Trits and words

A **trit** is −1, 0 or +1 (written `T`, `0`, `1`). A **word** is 27 trits,
holding every integer from −3,812,798,742,493 to +3,812,798,742,493.
Balanced ternary has no sign bit: a number is negative when its leading
non-zero trit is −1, and **negation is trit-by-trit inversion**. That
is why swapping the colours of the lamps negates the number.

Each row below shows the lamps, the decimal value, the balanced-ternary
digits and the *heptavintimal* notation (base 27, one letter per three
trits, nine letters per word) that the listings use.

In [2]:
panel.show_words([("1", 1), ("-1", -1), ("2 = 1T", 2), ("13", 13), ("-13", -13),
                  ("1967", 1967), ("-1967", -1967), ("3^13", 3**13),
                  ("largest", ternary.WMAX), ("smallest", ternary.WMIN)])

Truncating trits **rounds to the nearest integer**; it never chops toward
zero or toward minus infinity as binary does. Here is 1967 shifted right one
trit at a time: each step divides by 3, rounded.

In [3]:
panel.show_words([(f">> {k}", ternary.shift(1967, -k)) for k in range(8)])

## 2. The instruction word

An instruction is one word:

```
| op : 5 | r : 2 | x : 2 | m : 2 | addr : 16 |
```

These are an opcode, a register, an index register, an addressing mode
(−1 indirect, 0 direct, +1 immediate) and a 16-trit address. The factory
opcodes are all positive; the negative half is left free for instructions
the machine invents for itself (section 7).

In [4]:
for text in ["LD   R1, #12", "ADD  R1, 2(FP)", "ST   R2, @X", "J3   Y", "SEL  R1, #-11"]:
    display(panel.show_instruction(text))

field,trits,value,meaning
op,00001,1,LD
r,01,1,R1
x,00,0,none
m,01,1,immediate
addr,0000000000000110,12,12


field,trits,value,meaning
op,00011,4,ADD
r,01,1,R1
x,T1,-2,FP
m,00,0,direct
addr,000000000000001T,2,2


field,trits,value,meaning
op,0001T,2,ST
r,1T,2,R2
x,00,0,none
m,0T,-1,indirect
addr,0000000001101001,1000,1000


field,trits,value,meaning
op,01000,27,J3
r,00,0,R0
x,00,0,none
m,00,0,direct
addr,0000000010T1T01T,2000,2000


field,trits,value,meaning
op,01T00,18,SEL
r,01,1,R1
x,00,0,none
m,01,1,immediate
addr,0000000000000TT1,-11,-11


## 3. The micro-engine at work: a ternary multiplication

The Model 30 has no multiplier. `MUL` is a *microprogram* that walks the
multiplier one trit at a time. Each trit is −1, 0 or +1, so each pass
**subtracts, skips or adds** the shifted multiplicand. The multiplier is
never recoded, and the loop stops as soon as what remains of it is zero.

The program below computes 12 × −5. In the panel, press **step instr**
until the `MUL` is executing (the program listing highlights it), then
**step µ** through it and watch:

* **CNT** holds the multiplier, 12 = `110` in balanced ternary. It loses
  one trit per pass: 12 → 4 → 1 → 0.
* **U** holds the multiplicand, and triples each pass: −5, −15, −45.
* **T** accumulates the product: skip, add, add, according to the trits of
  12 read from the right (0, then 1, then 1).
* The microprogram listing jumps three ways at each `CASE C`: the
  sequencer itself is ternary.
* The datapath shows which register drives the **A bus** (green) and the
  **B bus** (blue), what the ALU does, and where the **D bus** (amber)
  delivers the result. Core reads flash green, writes red.

In [5]:
MUL_DEMO = '''
        ENTRY START
START:  LD   R1, #12        ; multiplier
        LD   R2, #-5        ; multiplicand
        MUL  R1, #0(R2)     ; R1 <- 12 x -5, by the microprogram
        ST   R1, RESULT
        PUSH R1
        POP  R3
        HLT
RESULT: DATA 0
'''
mul = panel.record_triad(MUL_DEMO, title="12 x -5 on the Model 30")
print("R1 =", mul.machine.rf[1], "  micro-cycles recorded:", len(mul.rec.upc))
panel.animate(mul, speed=30)

R1 = -60   micro-cycles recorded: 53


What each instruction cost, measured from the recording. A core
reference adds four cycles of wait, which is why the fetch alone costs six.

In [6]:
panel.show_costs(mul)

instruction,executed,micro-cycles (average),fewest,most
LD,2,10.0,10,10
MUL,1,24.0,24,24
ST,1,15.0,15,15
PUSH,1,13.0,13,13
POP,1,13.0,13,13


## 4. Where things live in core

Core holds 3¹² = 531,441 words, addressed **symmetrically** from
−265,720 to +265,720. The negative half belongs to the Executive, the
resident operating system, and is protected: a user program that touches
it takes a protection trap. A compiled SALISH program is laid out like
this:

* the **code**, from word 100;
* the **globals, strings and literals**;
* the **heap**, which grows upward from the end of the program;
* the **stack**, which grows downward from the top of core.

In [7]:
FACTORIAL = '''
-- factorials, printed in decimal and in balanced ternary
global facts[8]
proc fact(n) = n <= 1 -> 1, n * fact(n - 1)
proc main()
begin
  for i := 1 to 6 do begin
    facts[i] := fact(i)
    print(i); prints("! = "); print(facts[i]); prints(" = "); printt(facts[i]); newline()
  end
end
'''
asm, comp = salish.compile_source(FACTORIAL, "factorial.sal")
obj = triad.assemble(asm, origin=satellite.USER_ORIGIN)
panel.show_memory_map(obj)

The first lines of the machine code the compiler produced:

In [8]:
print("\n".join(obj.listing[:24]))

     100  CC00000EN  START:  CALL P_main
     101  DR0C00000  SVC  0
     102             P_fact:
     102  CHI000000  PUSH FP
     103  0HH000000  LEA  FP, 0(SP)
     104  0CG00000B  LD   R1, 2(FP)
     105  ACIC0000A  CMP  R1, #1
     106  CO00000DA  JP   L2
     107  0CIC0000A  LD   R1, #1
     108  BF00000DI  JMP  L3
     109             L2:
     109  0CG00000B  LD   R1, 2(FP)
     110  AOIC0000A  SUB  R1, #1
     111  CII000000  PUSH R1
     112  CC00000DU  CALL P_fact
     113  CII000000  PUSH R1
     114  0CG00000B  LD   R1, 2(FP)
     115  CMR000000  POP  R2
     116  ARKC00000  MUL  R1, #0(R2)
     117             L3:
     117  BF00000DK  JMP  L1
     118  0CIC00000  LD   R1, #0
     119             L1:
     119  0IP000000  LEA  SP, 0(FP)


## 5. A whole job: the Executive, a supervisor call, and back

Now the full system. The recording starts at the Executive's first
micro-cycle and runs to the halt. Things to find in the panel (the events
box lists the traps, and the slider jumps straight to them):

1. **Boot.** The Executive writes its greeting to the operator's console
   typewriter.
2. **The job arrives.** The Executive asks the satellite for work. The
   channel loads the program into core, and the *program* map fills.
3. **Into user mode.** An `RTI` starts the program. The **MODE** lamp turns
   blue (user), and `CALL P_main` pushes a return address onto the stack.
4. **Recursion.** `fact` calls itself. Watch **SP** walk down and the
   stack window fill with frames.
5. **A supervisor call.** To print, the program executes `SVC 1`. The
   micro-engine takes a trap: it writes the resume address, trap code and
   mode into the protected **trap words** (they flash red), switches MODE
   to supervisor and jumps into the Executive. The Executive saves the
   registers, sends the character to the printer with `OUT`, restores the
   registers and returns with `RTI`.
6. **Exit.** `SVC 0` reports the result to the satellite, and the Executive
   halts when there is no more work.

It is about 66,000 micro-cycles. Use the slider, or a high speed.

In [9]:
job = panel.record_salish(FACTORIAL, title="factorial job, Executive and all", max_frames=80000)
print("micro-cycles:", len(job.rec.upc), "  printer output:")
print("".join(chr(c) for f, c in job.rec.output))
panel.animate(job, speed=60)

micro-cycles: 66070   printer output:
DUWAMISH EXECUTIVE 1.0 READY
1! = 1 = 1
2! = 2 = 1T
3! = 6 = 1T0
4! = 24 = 10T0
5! = 120 = 11110
6! = 720 = 1000T00
DUWAMISH EXECUTIVE: NO MORE WORK. HALT.



### The same job, measured

The trap events, and what the instructions cost across the whole job:

In [10]:
for f, e in job.rec.events[:6]:
    print(f"{f:7}  {e}")
print("...")
panel.show_costs(job)

   2549  TRAP 6 (SUPERVISOR CALL) at 349, argument 1
   2899  TRAP 6 (SUPERVISOR CALL) at 220, argument 1
   3138  TRAP 6 (SUPERVISOR CALL) at 220, argument 1
   3377  TRAP 6 (SUPERVISOR CALL) at 220, argument 1
   3616  TRAP 6 (SUPERVISOR CALL) at 220, argument 1
   5083  TRAP 6 (SUPERVISOR CALL) at 349, argument 1
...


instruction,executed,micro-cycles (average),fewest,most
LD,3394,13.7,10,14
ST,1465,15.0,15,15
CMP,1037,11.7,10,14
ADD,592,10.6,10,14
JMP,554,10.0,10,10
PUSH,532,13.0,13,13
POP,459,13.0,13,13
JNP,332,11.0,11,11
JP,331,11.0,11,11
JZ,253,11.0,11,11


## 6. Two models of one architecture, and the Beer monitor

As with IBM's System/360, the architecture exists twice.

* The **Model 30** interprets every instruction with its microprogram.
* The **Model 90**, Cray's hardwired design, executes instructions
  directly.

They compute identical results; only the time differs. Every instruction
fetch also increments a hardware **tally** for its word of core: Stafford
Beer's idea that a system should watch itself. Here the tallies are read
out per procedure.

In [11]:
def run(model):
    sat = satellite.run_program("programs/hello.sal", model=model, out=lambda s: None)
    return sat
m30, m90 = run(30), run(90)
for name, sat in (("Model 30", m30), ("Model 90", m90)):
    how, value, where, cycles, instrs, secs = sat.jobs[0].steps[0].result
    print(f"{name}: {instrs:,} instructions, {cycles:,} cycles = {cycles*0.2/1000:.1f} ms of 1967 time")
print("same printout:", m30.jobs[0].steps[0].output == m90.jobs[0].steps[0].output)
panel.show_tally(m90.machine, m90.jobs[0].steps[0].obj)

Model 30: 47,329 instructions, 604,075 cycles = 120.8 ms of 1967 time
Model 90: 47,329 instructions, 236,570 cycles = 47.3 ms of 1967 time
same printout: True


## 7. The machine that rewrites its own microcode

This is the hardware behind the intelligence-explosion demonstration
(`programs/good/explosion.sal`). With the control-store key turned, a
program may write microcode. The loop below adds 10 + 9 + … + 1. We then
do what the explosion program does:

* copy the loop's three instructions into the fast **constant store**;
* write three microinstructions. Each is `IR <- K[k], PC+1, DISP next`: it
  loads a remembered instruction straight into IR and dispatches on it, as
  if it had just been fetched, but **without a core cycle**;
* point unused opcode **−1** at them, and patch the loop to use it.

In the second panel, step through a pass of the loop. The datapath shows
the **K store** driving IR, and no READ lamp lights for the second and
third instructions of the fused run. The microprogram listing shows the
invented microinstructions in green.

In [12]:
LOOP = '''
        ENTRY START
START:  LD   R1, #0
        LD   R2, #10
LOOP:   ADD  R1, #0(R2)
        SUB  R2, #1
        JP   LOOP
        HLT
'''
before = panel.record_triad(LOOP, title="the loop, as written")
after = panel.record_triad(LOOP, title="the loop, as a new instruction", wcs=True,
                           setup=lambda m, obj: panel.fuse(m, obj.symbols["LOOP"], 3))
for name, r in (("as written", before), ("fused", after)):
    print(f"{name:10}  R1 = {r.machine.rf[1]}   {r.machine.clock} micro-cycles")
display(panel.show_costs(after))
panel.animate(after, speed=20)

as written  R1 = 55   338 micro-cycles
fused       R1 = 55   248 micro-cycles


instruction,executed,micro-cycles (average),fewest,most
OP-1,10,22.0,22,22
LD,2,10.0,10,10


## 8. The floating-point feature

The Model 90 has a floating-point unit as standard, and the Model 30 can
be fitted with one. A float fills one word: a 5-trit exponent over a
22-trit mantissa, value = m × 3^(e−21). Both fields are balanced, so there
is no sign bit and no bias. The word is the sum e·3²² + m, which means **the
sign of a float is the sign of its mantissa, not of the word**. Here −2.0
is stored as a *positive* word, because its exponent is positive, and 1/3
as a *negative* one:

In [13]:
for x in (2.0, -2.0, 1/3, -1e-6):
    w = fpu.from_float(x)
    m, e = fpu.unpack(w)
    print(f"{x:>8.4g}   exponent {e:4}   mantissa {m:15,}   word {w:18,}")
panel.show_words([("2.0", fpu.from_float(2.0)), ("-2.0", fpu.from_float(-2.0)),
                  ("1/3", fpu.from_float(1/3)), ("-1e-6", fpu.from_float(-1e-6))])

       2   exponent    1   mantissa   6,973,568,802   word     38,354,628,411
      -2   exponent    1   mantissa  -6,973,568,802   word     24,407,490,807
  0.3333   exponent   -1   mantissa  10,460,353,203   word    -20,920,706,406
  -1e-06   exponent  -12   mantissa  -5,559,060,567   word   -382,131,775,875


On the Model 30 each floating-point order is two lines of microcode: fetch
the operand as usual, then one micro-order, `FPU`, hands R and MDR to the
unit. Watch the **FPU** control lamp. Without the feature, that micro-order
takes program check 11 instead.

In [14]:
CIRCLE = '''
proc main()
begin
  var r := float(7), pi := fdiv(float(355), float(113))
  print(fix(fmul(fmul(pi, r), r))); newline()    -- the area of a circle
end
'''
fp = panel.record_salish(CIRCLE, title="Model 30 with the floating-point feature", fpu=True)
print("".join(chr(c) for f, c in fp.rec.output))
panel.animate(fp, speed=60)

DUWAMISH EXECUTIVE 1.0 READY
154
DUWAMISH EXECUTIVE: NO MORE WORK. HALT.



In [15]:
bare = panel.record_salish(CIRCLE, title="Model 30 without it")
print("".join(chr(c) for f, c in bare.rec.output))

DUWAMISH EXECUTIVE 1.0 READY

*** PROGRAM CHECK 11 AT LOCATION 105, ARGUMENT 51
DUWAMISH EXECUTIVE: NO MORE WORK. HALT.



## 9. The compiler that compiles itself

`programs/selfhost/salish.sal` is SALISH/S: the SALISH compiler, written in
SALISH. It reads a program from the card reader and punches TRIAD code on
the card punch. The code is the same, card for card, as the compiler
that runs on the satellite. `jobs/bootstrap.job` runs four steps:

1. **Generation 0**, compiled by the satellite, compiles SALISH/S itself.
2. The cards it punched are assembled into **generation 1**, which
   compiles SALISH/S again.
3. **Generation 2** compiles a small program.
4. The small program runs.

Because generation 0 punches exactly the code it was made from,
generation 1 *is* generation 0, and the loop closes on itself at once.
This is Hofstadter's strange loop, a program that reproduces itself one
level up. It is also Good's machine that can build its own successor. It
takes about a minute and a half.

In [16]:
boot = satellite.run_decks(["jobs/bootstrap.job"], model=90, out=lambda s: None)
for i, st in enumerate(boot.jobs[0].steps, 1):
    for m in st.messages:
        print(f"step {i}: {m}")
panel.show_bootstrap(boot)

step 2: TRIAD: the deck punched by step 1: 15,154 cards, 168,330 words; identical, card for card, to the satellite's own compilation of ../programs/selfhost/salish.sal
step 3: TRIAD: the deck punched by step 2: 15,154 cards, 168,330 words; identical, card for card, to the deck of step 1; identical, card for card, to the satellite's own compilation of ../programs/selfhost/salish.sal
step 4: TRIAD: the deck punched by step 3: 858 cards, 838 words


The first cards generation 0 punched for itself:

In [17]:
print("\n".join(boot.jobs[0].steps[0].punched.splitlines()[:24]))

; SALISH compiler output
        ENTRY START
START:  CALL P_main
        SVC  0
; proc outc(c)
P_outc:
        PUSH FP
        LEA  FP, 0(SP)
        NEG  R1, G_buffering
        TST  #0(R1)
        JNP  L10
        LD   R1, 2(FP)
        SVC  5
        LD   R1, #0
        JMP  L9
L10:
        LD   R1, 2(FP)
        CMP  R1, #10
        JNZ  L11
        LD   R1, G_nlines
        CMP  R1, #1800
        JN   L12
        LD   R1, #S110
        PUSH R1


### The compiler improves itself, once

SALISH/S also carries the optimiser. With `-- OPT` on its first card it
compiles exactly as SALISH/O does: registers by usage counts, fixed
vectors, rotated loops, the peephole pass. So the plain generation 0 can
compile an *optimised* generation 1 of itself. `jobs/improve.job` does
that on the Duwamish (about four minutes):

| step | compiler | instructions to compile SALISH/S | its deck |
|---|---|---:|---|
| 1 | generation 0 (plain) | 104.0 million | 13,378 cards, the satellite's optimised compilation, card for card |
| 2 | generation 1 (optimised) | 87.2 million | identical |
| 3 | generation 2 | 87.2 million | identical |

Generation 1 is 16% faster and thinks exactly the same thoughts, so its
successor is itself. The improvement happens once and stops: a fixed
point. The cell below shows the same thing in seconds. Generation 1 is
taken from the satellite (the tests check it is identical to what
generation 0 punches). Then both generations compile a few programs.

In [18]:
def compile_with(generation_opt, program):
    deck = ("//JOB G\n//SALISH FROM=programs/selfhost/salish.sal"
            + (" OPT" if generation_opt else "") + "\n//EXEC\n//DATA\n-- OPT\n"
            + f"//DATA FROM={program}\n//DATA FROM=duwamish/lib/runtime.sal\n")
    sat = satellite.Satellite([(deck, "g.job")], model=90, out=lambda s: None)
    sat.run()
    st = sat.jobs[0].steps[0]
    return st.punched, st.result[4]
rows = []
for prog in ("programs/hello.sal", "programs/kleene.sal", "programs/hofstadter/miu.sal"):
    d0, n0 = compile_with(False, prog)
    d1, n1 = compile_with(True, prog)
    print(f"{prog:32} same deck: {d0 == d1}   generation 0: {n0:,}   generation 1: {n1:,}"
          f"   ({100 * (n0 - n1) / n0:.0f}% fewer)")
    rows.append((prog.split("/")[-1], [n0, n1]))
panel.show_bars("instructions to compile, optimising", rows,
                ["generation 0 (plain)", "generation 1 (optimised)"])

programs/hello.sal               same deck: True   generation 0: 5,202,951   generation 1: 4,354,500   (16% fewer)


programs/kleene.sal              same deck: True   generation 0: 5,726,882   generation 1: 4,787,576   (16% fewer)


programs/hofstadter/miu.sal      same deck: True   generation 0: 9,684,906   generation 1: 8,107,201   (16% fewer)


## 10. Copycat: analogy as perception

"If abc changes to abd, what does ijk change to?" Hofstadter and
Mitchell's Copycat answers such questions with many small, random agents
(codelets). The agents notice bonds between letters, group alike letters,
describe the change as a rule, and map it across, letting concepts
*slip*: rightmost into leftmost, successor into predecessor, letter into
group. A *temperature* measures how incoherent the current understanding
is. `programs/hofstadter/copycat.sal` is a cut-down version on the
Duwamish. Each problem is run 30 times. The bars show how often each
answer came up, coloured by the average temperature at which it was
found. A cool answer is one the program itself found coherent. Look at
xyz: z has no successor, and the snag leads some runs to the mirror
answer wyz. It is rarer than the literal xyd, and cooler. The run takes
about half a minute.

In [19]:
cc = satellite.run_program("programs/hofstadter/copycat.sal", model=90,
                           optimise=True, out=lambda s: None)
panel.show_copycat(cc.jobs[0].steps[0].output)

## 11. Good's five-year plan: where the search goes

Shannon's chess program looks the same distance down every line. Good
wanted a program to follow the lines that are *likely to be played* and
cut the unlikely ones short. `programs/good/fiveyear.sal` gives every
move a plausibility and follows a line while its probability stays above
a threshold. Here is one move's search from an opening. Shannon's player
cuts every line at the second ply. Good's player cuts implausible lines
at the first and follows plausible ones four or five plies deep, for
about the same number of positions. `jobs/fiveyear.job` plays the full
match.

In [20]:
FIVE = open("programs/good/fiveyear.sal").read()
FIVE = FIVE.replace('get "chessbase"', 'get "programs/good/chessbase.sal"')
FIVE = FIVE[:FIVE.index("proc main()")] + '''
proc main()
begin
  var col := 1
  setup()
  seed := 1975
  for j := 1 to 4 do begin random_move(col); col := -col end
  think(1, col)
  prints("shannon "); print(nodes); newline()
  think(2, col)
  prints("good "); print(nodes); newline()
  for p := 1 to 12 do begin print(cutoff[p]); space() end
  newline()
end
'''
import tempfile
with tempfile.NamedTemporaryFile("w", suffix=".sal", dir=".", delete=False) as f:
    f.write(FIVE)
fy = satellite.run_program(f.name, model=90, optimise=True, out=lambda s: None)
os.unlink(f.name)
lines = fy.jobs[0].steps[0].output.split("\n")
print(lines[0], "positions;", lines[1], "positions")
cuts = [int(x) for x in lines[2].split()]
panel.show_bars("lines cut off, by ply (Shannon's player: all at ply 2)",
                [(f"ply {p}", [c]) for p, c in enumerate(cuts, 1) if c],
                ["Good's player"])

shannon 334 positions; good 401 positions


## 12. The look-ahead unit

The Model 90 issues one instruction per 1 µs core cycle. The optional
look-ahead unit adds a 32-word **instruction stack**, after the CDC 6600:
a loop that fits in it runs without fetching instructions from core. An
instruction from the stack that makes no data reference to core takes
one 200 ns cycle. Here are the Livermore kernels, compiled by SALISH/O
and run with the floating-point unit, without and with the look-ahead
unit. The answers are identical; only the time changes.

In [21]:
import re
def livermore(lookahead):
    sat = satellite.run_decks(["jobs/livermore.job"], model=90,
                              lookahead=lookahead, out=lambda s: None)
    out = sat.jobs[1].steps[0].output          # the job compiled by SALISH/O
    rows = re.findall(r"^\s*(\d+)\s+(\S.*?\S)\s+\d+\s+\d+\s+[\d.]+\s+\d+\s+[\d.]+"
                      r"\s+\d+\s+([\d.]+)", out, re.M)
    return {f"{k} {name}": float(v) for k, name, v in rows}
plain, fast = livermore(False), livermore(True)
panel.show_bars("Livermore kernels, hardware float, SALISH/O (MFLOPS)",
                [(k, [plain[k], fast[k]]) for k in plain],
                ["Model 90", "Model 90 + look-ahead"], fmt="{:.3f}")

## 13. TRILISP's memory: allocation and garbage collection

TRILISP keeps its list cells in three vectors in core: `car_`, `cdr_` and
`mark_`. The free cells are chained through `cdr_`. When the chain runs
out, `cons` calls the collector, which works in two phases.

* **Mark.** It marks every cell reachable from the symbols, and from any
  word on the machine stack that looks like a pointer. This is a
  *conservative* collector.
* **Sweep.** It sweeps the heap from the top down, clearing the marks
  and chaining every unmarked cell back onto the free list.

Here TRILISP runs with a heap of 3⁷ = 2,187 cells, so that collections
come often, on a program that builds lists and throws most of them away.
It keeps a few in `KEPT`, which survive every collection. The recorder
reads the heap straight out of core every few hundred instructions:
* cells light **amber** as `cons` hands them out;
* they turn **green** as the collector marks the live ones;
* the sweep reclaims the rest, and they go dark.

The lower plot is the number of cells in use, against instructions
executed. It is a sawtooth: each drop is one collection. The recording
takes a few seconds.

In [22]:
from duwamish import heapview
print(heapview.WORKLOAD)
heap = heapview.record_heap()
print(heap.output.strip().splitlines()[-1])
heapview.animate(heap)


(DEFINE BUILD (LAMBDA (N) (COND ((EQ N 0) NIL) (T (CONS N (BUILD (- N 1)))))))
(DEFINE KEPT NIL)
(DEFINE CHURN (LAMBDA (K)
  (COND ((EQ K 0) (LENGTH KEPT))
        (T (PROGN (SETQ KEPT (CONS (BUILD 8) KEPT))
                  (BUILD 120)
                  (CHURN (- K 1)))))))
(CHURN 14)



END OF CARDS.  27387 EVALUATIONS, 18325 CONSES, 10 GARBAGE COLLECTIONS.


## 14. TRI-TRAN: FORTRAN on a ternary machine

The laboratories who were to buy the Duwamish would bring FORTRAN, so the
machine has a FORTRAN of 1966: TRI-TRAN (`docs/TRITRAN.md`). Its LOGICAL
is three-valued, and `.AND.`, `.OR.` and `.EQV.` are single Kleene
instructions. Its arithmetic IF, `IF (E) 10, 20, 30`, is one `J3`. Like
FORTRAN I, it keeps each DO loop's index in an index register. Here is a
small program and the code the compiler makes for it (without its
library):

In [23]:
from duwamish import tritran
SOURCE = '''
      LOGICAL P, Q
      DIMENSION A(100)
      P = .TRUE.
      Q = .UNKNOWN.
      PRINT 10, P .AND. Q, P .OR. Q, .NOT. Q
   10 FORMAT (1X, 3L2)
      DO 20 K = 1, 100
   20 A(K) = FLOAT(K - 50)
      M = 0
      N = 0
      DO 30 K = 1, 100
        IF (A(K)) 21, 22, 30
   21   M = M + 1
        GO TO 30
   22   N = K
   30 CONTINUE
      PRINT 40, M, N
   40 FORMAT (1X, I3, ' NEGATIVE; ZERO AT', I4)
      END
'''
asm, comp = tritran.compile_source(SOURCE)
print(asm.split("; SALISH compiler output")[0])
sat = satellite.Satellite([("//JOB T\n//TRITRAN\n" + SOURCE + "\n//EXEC\n", "nb.job")],
                          model=90, out=lambda s: None)
sat.run()
print(sat.jobs[0].steps[0].output)

; TRI-TRAN compiler output
        ENTRY START
START:  CALL P_tt_init
        CALL _MAIN
        LD   R1, #0
        SVC  0
; main program   index registers: R3=K
_MAIN:
        LD   R1, #1
        ST   R1, _MAIN.P
        LD   R1, #0
        ST   R1, _MAIN.Q
        LEA  R1, _MAIN.10
        PUSH R1
        LD   R1, #6
        PUSH R1
        CALL P_tt_wbeg
        LD   R1, _MAIN.P
        AND  R1, _MAIN.Q
        PUSH R1
        CALL P_tt_wl
        LD   R1, _MAIN.P
        OR   R1, _MAIN.Q
        PUSH R1
        CALL P_tt_wl
        NEG  R1, _MAIN.Q
        PUSH R1
        CALL P_tt_wl
        CALL P_tt_wend
        LD   R3, #1
_MAIN._2:
_MAIN.20:
        LD   R1, #0(R3)
        SUB  R1, #50
        FLT  R1, #0(R1)
        ST   R1, _MAIN.A-1(R3)
        ADD  R3, #1
        CMP  R3, #100
        JNP  _MAIN._2
        ST   R3, _MAIN.K
        LD   R1, #0
        ST   R1, _MAIN.M
        LD   R1, #0
        ST   R1, _MAIN.N
        LD   R3, #1
_MAIN._3:
        LD   R1, _MAIN.A-1(R3)


The same six Livermore kernels, written in FORTRAN
(`programs/livermore.ftn`), compute the same words to the last trit as
the SALISH program's hardware column. Here is how fast each compiler's
code runs them on the Model 90 with the floating-point unit. FORTRAN I's
one idea, the index register, takes TRI-TRAN from plain SALISH's speed
to within 1% of SALISH/O overall.

In [24]:
deck = '''//JOB A
//SALISH FROM=programs/livermore.sal
//EXEC
//JOB B
//TRITRAN FROM=programs/livermore.ftn
//EXEC
//JOB C
//SALISH FROM=programs/livermore.sal OPT
//EXEC
'''
sat = satellite.Satellite([(deck, "nb.job")], model=90, out=lambda s: None)
sat.run()
def salish_rates(out):
    rows = re.findall(r"^\s*(\d+)\s+(\S.*?\S)\s+\d+\s+\d+\s+[\d.]+\s+\d+\s+[\d.]+"
                      r"\s+\d+\s+([\d.]+)", out, re.M)
    return [float(v) for k, name, v in rows]
plain = salish_rates(sat.jobs[0].steps[0].output)
fortran = [float(l.split()[3]) for l in sat.jobs[1].steps[0].output.splitlines()
           if len(l.split()) == 5 and l.split()[0].isdigit()]
best = salish_rates(sat.jobs[2].steps[0].output)
names = ["1 hydro", "3 inner product", "5 tri-diagonal", "7 equation of state",
         "11 first sum", "12 first difference"]
panel.show_bars("Livermore kernels on the Model 90 with the FPU (MFLOPS)",
                [(n, [a, b, c]) for n, a, b, c in zip(names, plain, fortran, best)],
                ["plain SALISH", "TRI-TRAN", "SALISH/O"], fmt="{:.3f}")

## 15. Rosenblatt's perceptron, and Good's weight of evidence

Rosenblatt's perceptron, in ternary (`programs/good/perceptron.sal`,
and `docs/DEMONSTRATOR.md` §10). A 7 × 7 retina of ink (+1), paper (−1)
and *unseen* (0) points; 243 association units wired at random to three
nearby points by excitatory (+1) and inhibitory (−1) connections; a
response of +1, −1 or 0, "I do not know". It learns to tell a vertical bar
from a horizontal one by error correction. Against it is Good's method
with the same units: count, in one pass, how each unit's testimony goes
with each class, and add weights of evidence in decibans, answering only
past 20 db. Then both are tested on new patterns, whole and with a third
of the retina hidden. The run takes about half a minute.

In [25]:
sat = satellite.Satellite([("//JOB P\n//SALISH FROM=programs/good/perceptron.sal OPT\n//EXEC\n",
                            "nb.job")], model=90, out=lambda s: None)
sat.run()
out = sat.jobs[0].steps[0].output
print(out[out.index("2. TESTING"):])
rows = []
for part, label in (("the whole retina seen", "seen"), ("a third of the retina unseen", "a third unseen")):
    text = out.split(part)[1].split("\n\n")[0]
    for name in ("perceptron   ", "cautious perceptron", "Good's weight of evidence"):
        line = [l for l in text.splitlines() if name in l][0]
        rows.append((f"{name.strip()}, {label}", [int(x) for x in line.split()[-3:]]))
panel.show_bars("200 new patterns: right, wrong, don't know", rows,
                ["right", "wrong", "don't know"])

2. TESTING, on 200 new patterns of each kind

                                    right  wrong  don't know
   the whole retina seen
     perceptron                     176     19          5
     cautious perceptron            150      7         43
     Good's weight of evidence      167      8         25
       (its surest wrong answer claimed +41.3 db)
   a third of the retina unseen
     perceptron                     121     77          2
     cautious perceptron            104     53         43
     Good's weight of evidence      122     19         59
       (its surest wrong answer claimed +65.2 db)

3. THE LIMIT: is the number of ink spots among five points odd?

     81 A-units, each looking at k of the five points; the
     32 patterns shown again and again, up to 100 times

         k                                right  wrong  don't know
         1      not learned in 100 passes        0      0         32
         2      not learned in 100 passes        0      0         32
 

## 16. Load your own program

Edit the SALISH source below and run the cell. `record_salish` also accepts
a path, for example `"programs/hofstadter/selfref.sal"`, but long
programs make long recordings. Keep `max_frames` to a few hundred thousand.

`panel.save_html(recording, "panel.html")` writes the front panel as a
stand-alone page that any browser can open.

In [26]:
MY_PROGRAM = '''
proc main()
begin
  var x := 1
  for i := 1 to 5 do begin
    x := x * 3
    printt(x); newline()        -- powers of three: 1, 10, 100, ... in ternary
  end
end
'''
mine = panel.record_salish(MY_PROGRAM, title="my program", max_frames=100000)
print("".join(chr(c) for f, c in mine.rec.output))
panel.animate(mine, speed=80)

DUWAMISH EXECUTIVE 1.0 READY
10
100
1000
10000
100000
DUWAMISH EXECUTIVE: NO MORE WORK. HALT.



---
*See `docs/ARCHITECTURE.md` for the machine, `docs/SALISH.md`,
`docs/TRILISP.md` and `docs/TRITRAN.md` for the languages, and `docs/DEMONSTRATOR.md` for the
demonstrations of Good's and Hofstadter's ideas.*